# Models that learn {#sec-models-that-learn}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c11-models-that-learn.ipynb)

A data scientist joins the waiting-list team and asks a fair question: why build a stock-and-flow model at all, when a forecasting model trained on ten years of data could predict the list? This chapter answers it with experiments rather than opinions. Machine learning and artificial intelligence can help at every stage of a system dynamics project, from reading interviews to searching policies. They also fail in specific, predictable ways, and the most important one is a change of policy: exactly the situation decision makers ask about.

```
                     Effect of pressure (learned from data)
                                   |
                                   v
Referrals ----> [ Waiting list ] ----> Treatments = Capacity x Effect
 (observed)            |                     ^
                       +--- Pressure --------+
```

*A hybrid model from section 3: the stock, the flows and the units come from structure; one nonlinear effect is learned from data.*

The chapter is organised by the stages of the modelling life cycle: problem articulation and conceptualisation (section 1), formulation (section 2), parameter estimation (section 3), testing (section 4), and policy and use (section 5). Section 6 is a caution about overfitting, leakage and explainability. No home molecules are introduced; the molecules used come from earlier chapters and, like all in this book, are after Hines, *Molecules of Structure* [@hines_molecules].

**Who it is for.** System dynamicists curious about machine learning, and data scientists curious about system dynamics. **Who can skip.** Nobody should skip section 4; readers short of time may skim sections 1 and 5. **Prerequisites.** Chapters 1 to 4 and chapter 9 (Data, calibration and uncertainty); least squares with numpy. **Time.** About three hours.

**Learning outcomes.** By the end of this chapter you can:

1. **design** a prompt that asks a large language model for a causal draft with evidence, and **check** the draft against the source text (section 1, Try it);
2. **recover** a known molecule's equation from noisy simulated data by sparse regression (section 2, Try it);
3. **build** a hybrid model whose nonlinear effect is fitted to data inside a stock-and-flow structure (section 3, Try it);
4. **evaluate** a pure data model against a system dynamics model before and after a policy change, and **explain** the difference (section 4, Try it);
5. **build** and validate a surrogate model for fast policy search (section 5, Try it);
6. **diagnose** overfitting and leakage in a forecasting experiment (section 6, Try it).

In [ ]:
# Run this cell first. In Colab it installs loopbricks (about ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 11
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours, each with its own line style, so figures read in greyscale
OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00",
             "#0072B2", "#CC79A7"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = cycler(color=OKABE_ITO) + \
    cycler(linestyle=STYLES)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__)

## 1. Problem articulation: a language model as drafting assistant

Early in a project the modeller reads interviews, notes and reports, and drafts a list of variables and a causal loop diagram (chapter 2, From story to diagram). A **large language model** (LLM) can produce such a draft in seconds. The risk is just as fast: a fluent draft can contain links nobody said, quotes nobody gave, and polarities that are backwards. The remedy is to ask for evidence with every link, and then check the evidence by code, not by eye.

Here are four short interview notes from the running case. They are invented for the book.

In [ ]:
interviews = {
    "consultant": "When the list gets very long we run extra Saturday "
    "clinics, but my team cannot keep that up for more than a few months "
    "before people burn out.",
    "GP": "If I know the wait is over a year, I sometimes refer to the "
    "private clinic instead. Other times I refer earlier than I would, "
    "just to get the patient in the queue.",
    "manager": "Every time we validate the list we find patients who were "
    "treated elsewhere or no longer need the appointment.",
    "advocate": "Patients who wait longer get sicker, and some of them end "
    "up needing more complex care, which takes longer to deliver.",
}
for who, text in interviews.items():
    print(f"{who:>10}: {text}")

A good prompt states the task, the format, and the rules of evidence. It asks for variables as quantities that can rise or fall, with units, and for every link a verbatim quote, or the word `DEFINITION` for stock-and-flow identities that need no quote. We show the prompt as text; this chapter makes no call to any LLM service.

In [ ]:
PROMPT = """You are helping a system dynamics modeller.
From the interview notes below, list:
1. Variables: quantities that can rise or fall, each with a unit.
2. Causal links: cause, effect, polarity (+ or -), and EVIDENCE, a
   verbatim quote of at most 20 words from the notes. If the link is a
   stock-and-flow identity (an inflow raises its stock), write DEFINITION.
   If you cannot quote the notes, write UNSUPPORTED. Never invent quotes.
Return JSON: {"variables": [...], "links": [...]}.

NOTES:
""" + "\n".join(f"[{k}] {v}" for k, v in interviews.items())
print(PROMPT)

Below is a draft of the kind such a prompt returns. We wrote it by hand so that the chapter runs anywhere, and we planted the three faults that real drafts most often contain: a quote that is not in the notes, an effect with two opposite polarities, and a link that rests on interpretation rather than on anything said.

In [ ]:
draft = [  # cause, effect, polarity, evidence
    ("Referrals", "Waiting list", "+", "DEFINITION"),
    ("Treatments", "Waiting list", "-", "DEFINITION"),
    ("Waiting list", "Waiting time", "+", "DEFINITION"),
    ("Waiting list", "Extra clinics", "+",
     "When the list gets very long we run extra Saturday clinics"),
    ("Extra clinics", "Treatments", "+", "we run extra Saturday clinics"),
    ("Extra clinics", "Staff fatigue", "+",
     "cannot keep that up for more than a few months before people burn"
     " out"),
    ("Staff fatigue", "Treatments", "-", "people burn out"),
    ("Waiting time", "Referrals", "-",
     "I sometimes refer to the private clinic instead"),
    ("Waiting time", "Referrals", "+", "I refer earlier than I would"),
    ("Validation", "Waiting list", "-",
     "we find patients who were treated elsewhere"),
    ("Waiting time", "Severity", "+", "Patients who wait longer get sicker"),
    ("Severity", "Treatment time", "+",
     "more complex care, which takes longer to deliver"),
    ("Treatment time", "Treatments", "-", "UNSUPPORTED"),
    ("Waiting time", "Complaints", "+",
     "patients complain to their local politicians every week"),
]


def norm(s):
    return " ".join(s.lower().split())


corpus = norm(" ".join(interviews.values()))
review = []
for cause, effect, sign, ev in draft:
    if ev == "DEFINITION":
        status = "identity: check against the stock-and-flow map"
    elif ev == "UNSUPPORTED":
        status = "no evidence: modeller's judgement, mark as assumption"
    elif norm(ev) in corpus:
        status = "quote found"
    else:
        status = "QUOTE NOT IN NOTES: reject or ask"
    review.append((f"{cause} -> {effect} ({sign})", status))
pairs = pd.Series([(c, e) for c, e, _, _ in draft])
for (c, e) in pairs[pairs.duplicated()]:
    review.append((f"{c} -> {e}", "TWO POLARITIES: split into two causes"))
print(pd.DataFrame(review, columns=["link", "check"]).to_string(index=False))

Of the 14 links, three are stock-and-flow identities, nine have quotes that are really in the notes, one (`Treatment time -> Treatments`) is the modeller's own assumption, and one quote was invented: nobody mentioned complaints or politicians. The check also caught the pair `Waiting time -> Referrals` drawn with both polarities. A string match is crude (a paraphrase fails it), but crude is the right default: a quote that cannot be found is treated as missing until a person finds it.

Loops matter more than single links, so the last check finds every feedback loop in the draft and labels it **reinforcing** (R, an even number of negative links) or **balancing** (B, an odd number). A short depth-first search does it.

In [ ]:
def loops(links):
    # every simple cycle in a list of (cause, effect, sign) links
    out = {}
    nodes = sorted({x for c, e, _ in links for x in (c, e)})
    nxt = {n: [(e, s) for c, e, s in links if c == n] for n in nodes}

    def walk(start, node, path, signs):
        for e, s in nxt[node]:
            if e == start:
                key = frozenset(zip(path, path[1:] + [start], signs + [s]))
                if key not in out:
                    neg = (signs + [s]).count("-")
                    out[key] = (" -> ".join(path + [start]),
                                "B" if neg % 2 else "R")
            elif e not in path and e > start:
                walk(start, e, path + [e], signs + [s])

    for n in nodes:
        walk(n, n, [n], [])
    return list(out.values())


kept = [(c, e, s) for c, e, s, ev in draft
        if ev == "DEFINITION" or norm(ev) in corpus]
for path, kind in loops(kept):
    print(kind, path)

Four loops survive the checks. Extra clinics relieve the list (B); the same clinics, by burning staff out, lower treatments and lengthen the list further (R); and waiting time feeds back on referrals both ways, through private care (B) and through early referrals (R), with identical nodes, which is why the two polarities must be separated before the diagram means anything. Note what is missing: the advocate's loop through severity and treatment time would close only through the unsupported link, so it stays a hypothesis to test with the clinicians.

The code did the tedious part: it confirmed quotes, flagged an invented one and a contradiction, and enumerated loops. It cannot judge whether a quote really supports a link ("people burn out" does not by itself say that fatigue lowers treatments), whether the variables are the right ones, or whether something important was never said. That remains the modeller's work, with the interviewees. Two practical rules: never send confidential interview material to an external service without consent and a data agreement, and record the prompt, the model version and the date with the draft, as you would for any other source.

**Try it.**

- *Check.* Which link in the draft has a quote that is in the notes but does not, by itself, support the claimed effect?
- *Apply.* The two opposite links from waiting time to referrals are both real. Rewrite them as two separate variables (for example `Referrals to private care` and `Early referrals`) and rerun the loop finder. Which loops do you get?
- *Extend.* Add a check that every variable named in a link appears in at least one link marked `quote found` or `DEFINITION`. Which variables fail?

<details><summary>Worked answer</summary>

*Check.* `Staff fatigue -> Treatments (-)` with the quote "people burn out". The quote says staff tire; that fatigue lowers the number of treatments is the drafter's inference, and a plausible one, but it should be confirmed with the consultant. (`Extra clinics -> Treatments` is similar: the quote shows the clinics exist, not how many patients they add.)

*Apply.* With the two causes split, the loop finder returns four loops: the balancing loop through `Private referrals`, the reinforcing loop through `Early referrals`, and the two loops through extra clinics as before. Each can now be given its own strength and delay in a model.

*Extend.* Only `Complaints` fails: it appears in the invented link and nowhere else, so it drops out with that link.

</details>

In [ ]:
print("Link resting on interpretation: Staff fatigue -> Treatments (-),"
      " quote 'people burn out'")
split = [(c, e, s) for c, e, s in kept if not (c == "Waiting time"
                                               and e == "Referrals")]
split += [("Waiting time", "Private referrals", "+"),
          ("Private referrals", "Referrals", "-"),
          ("Waiting time", "Early referrals", "+"),
          ("Early referrals", "Referrals", "+")]
for path, kind in loops(split):
    print(kind, path)
ok = {x for c, e, s, ev in draft if ev == "DEFINITION" or norm(ev) in corpus
      for x in (c, e)}
print("Variables without support:",
      sorted({x for c, e, _, _ in draft for x in (c, e)} - ok))

## 2. Formulation: discovering structure from data

Suppose you have a time series of a stock and suspect it obeys some simple rate equation, but you do not know which. **Sparse regression** turns the search into least squares. Estimate the rate of change $\dot x$ from the data; build a **library** of candidate terms (a constant, $x$, $x^2$, $x^3$, and so on); then find the fewest terms that explain $\dot x$:

$$\dot x \approx \Theta(x)\,\xi,\qquad \Theta(x) = [\,1,\ x,\ x^2,\ x^3\,],$$

keeping only the coefficients in $\xi$ that are clearly not zero. This is the idea of SINDy, sparse identification of nonlinear dynamics [@brunton2016]. The simplest version repeats two steps: fit by least squares, then set small coefficients to zero and refit with the rest.

We test it where we know the answer. The Constrained Growth molecule (`logistic`) from chapter 3 has

$$\frac{dP}{dt} = r P - \frac{r}{K}P^2,$$

with $r$ = 0.18 per year and $K$ = 200 (its defaults). We sample it every half year for 40 years, add 1% measurement noise, and ask the regression to find the equation.

In [ ]:
lg = lb.molecule("logistic")
t_obs = np.arange(0, 40.01, 0.5)                       # years
clean = lg.run(T=40).loc[t_obs, "Population"].to_numpy()


def derivative(x, t, width=5):
    # smooth with a moving average, then take central differences
    xs = np.convolve(x, np.ones(width) / width, mode="same")
    edge = width // 2
    xs[:edge], xs[-edge:] = x[:edge], x[-edge:]
    return xs[edge:-edge], np.gradient(xs, t)[edge:-edge]


def sindy(x, dx, names, degree=3, threshold=0.1, rounds=10):
    lib = np.column_stack([x ** k for k in range(degree + 1)])
    scale = np.linalg.norm(lib, axis=0)                 # comparable columns
    a = lib / scale
    xi = np.linalg.lstsq(a, dx, rcond=None)[0]
    dense = xi / scale
    for _ in range(rounds):
        small = np.abs(xi) < threshold * np.abs(xi).max()
        xi[small] = 0.0
        xi[~small] = np.linalg.lstsq(a[:, ~small], dx, rcond=None)[0]
    return pd.DataFrame({"least squares": dense, "sparse": xi / scale},
                        index=names)


names = ["1", "P", "P^2", "P^3"]
noisy = clean * (1 + rng.normal(0, 0.01, len(clean)))
x_s, dx = derivative(noisy, t_obs)
fit = sindy(x_s, dx, names)
print(fit.to_string(float_format=lambda v: f"{v: .6f}"))
b1, b2 = fit.loc["P", "sparse"], fit.loc["P^2", "sparse"]
print(f"\nRecovered r = {b1:.4f} per year (true 0.18), "
      f"K = {-b1 / b2:.1f} (true 200)")

Plain least squares puts some weight on every term, including a constant of about -0.013 and a cubic term too small to print. After thresholding only $P$ and $P^2$ remain, with coefficients 0.1792 and -0.000895, which give $r$ = 0.1792 per year and $K$ = 200.3 against the true 0.18 and 200.

In [ ]:
fig, ax = plt.subplots()
grid = np.linspace(0, 210, 200)
ax.plot(x_s, dx, "o", ms=4, label="estimated from noisy data")
ax.plot(grid, 0.18 * grid - 0.18 / 200 * grid ** 2, label="true equation")
ax.plot(grid, b1 * grid + b2 * grid ** 2, label="recovered equation")
ax.set(title="Rate of change against the stock: logistic growth",
       xlabel="Population (individuals)",
       ylabel="dP/dt (individuals/year)")
ax.legend()
plt.show()

*Figure: the rate of change of the population against the population. Dots are estimated from the noisy series; the recovered equation lies on top of the true one.*

Plotting the rate against the stock, a **phase plot**, is the most useful picture in structure discovery: a parabola through zero says "growth that limits itself" before any regression is run.

Two warnings. First, the method finds an equation, not a mechanism: $rP - (r/K)P^2$ is equally the logistic of chapter 3 and, for instance, a diffusion model whose potential adopters are $K-P$. The modeller still decides which story is right. Second, derivatives of noisy data are noisy, and noise is where the method breaks, as the exercise shows.

**Try it.**

- *Check.* Without the sparse step (the "least squares" column), how many terms have a coefficient that is not zero?
- *Apply.* Raise the noise to 3%. Do you still recover the right terms? How far off are $r$ and $K$?
- *Extend.* Apply the method to the Decay molecule (`decay`) from chapter 3 and confirm it finds a single linear term. Compare the coefficient with the molecule's own parameter.

<details><summary>Worked answer</summary>

*Check.* All four: least squares never sets a coefficient exactly to zero, which is why the sparse step is needed.

*Apply.* With 3% noise and this seed the same two terms are found, with $r$ = 0.1793 per year and $K$ = 200.7. Try other seeds: as noise grows, spurious terms such as $P^3$ start to survive the threshold, and the smoothing width and the threshold become choices you must report.

*Extend.* The Decay molecule's equation is `Decay = Stock / Decay_time` with a decay time of 5 years, so $\dot x = -0.2x$. The method keeps only the linear term, with a coefficient of -0.2033.

</details>

In [ ]:
print("Non-zero least-squares terms:",
      int((fit["least squares"].abs() > 1e-12).sum()))
noisy3 = clean * (1 + np.random.default_rng(SEED).normal(0, 0.03, len(clean)))
f3 = sindy(*derivative(noisy3, t_obs), names)
print(f3["sparse"].to_string(float_format=lambda v: f"{v: .6f}"))
print(f"3% noise: r = {f3.loc['P', 'sparse']:.4f}, "
      f"K = {-f3.loc['P', 'sparse'] / f3.loc['P^2', 'sparse']:.1f}")
dec = lb.molecule("decay")
print(dec.equations())
stock = dec.stocks[0]["name"]
dd = dec.run(T=40).loc[t_obs, stock].to_numpy()
fd = sindy(*derivative(dd * (1 + np.random.default_rng(SEED).normal(
    0, 0.01, len(dd))), t_obs), ["1", "x", "x^2", "x^3"])
print(fd["sparse"].to_string(float_format=lambda v: f"{v: .6f}"))
print("Decay parameters:", dec.params)

## 3. Parameter estimation: hybrid models

A **hybrid model** keeps the parts of the structure we are sure of (stocks, flows, units, conservation of patients) and learns from data only the part we are unsure of. In the waiting list, we are sure that the list is referrals in minus treatments out. We are less sure how treatments respond to pressure: when the list is long, staff run extra clinics and work faster, up to a limit (section 1's consultant said as much). That response is an **effect**, a dimensionless multiplier, the Effect of a Dimensionless Input (`effectFunction`) of chapter 5.

To test the method where we know the truth, we build a "true" world with a known effect table, generate ten years of monthly data from it, and then pretend we do not know the table. Unlike the published data, suppose the hospital's own records give monthly referrals and capacity.

In [ ]:
TRUE_EFFECT = "TABLE(Pressure, 0.5, 0.85, 1, 1, 1.5, 1.1, 2, 1.15, 3, 1.17)"


def pressure_model(effect_eq):
    return lb.Model.from_spec({
        "name": "Waiting list with pressure", "timeUnit": "month",
        "T": 120, "dt": 0.25,
        "stocks": [{"id": "wl", "name": "Waiting list", "init": "30000",
                    "units": "patients", "nonNegative": True}],
        "flows": [
            {"id": "r", "name": "Referrals", "to": "wl", "eq": "4000",
             "units": "patients/month"},
            {"id": "t", "name": "Treatments", "from": "wl",
             "eq": "MIN(Capacity * Effect_of_pressure, "
                   "Waiting_list / Minimum_treatment_time)",
             "units": "patients/month"}],
        "vars": [
            {"id": "c", "name": "Capacity", "eq": "4000",
             "units": "patients/month"},
            {"id": "rl", "name": "Reference list", "eq": "40000",
             "units": "patients"},
            {"id": "p", "name": "Pressure",
             "eq": "Waiting_list / Reference_list", "units": "1"},
            {"id": "e", "name": "Effect of pressure", "eq": effect_eq,
             "units": "1"},
            {"id": "m", "name": "Minimum treatment time", "eq": "0.5",
             "units": "month"}]})


months = np.arange(121.0)
referrals = pd.Series(4000 + 900 * np.sin(2 * np.pi * months / 60)
                      + np.cumsum(rng.normal(0, 40, 121)), index=months)
truth = pressure_model(TRUE_EFFECT).run(inputs={"Referrals": referrals})
observed = truth.loc[months, "Waiting list"].to_numpy() \
    * (1 + rng.normal(0, 0.005, 121))
print(truth.loc[[0.0, 30.0, 60.0, 120.0],
                ["Waiting list", "Pressure", "Treatments"]].round(2))

In the "true" world the list swings between about 30,000 and 49,000 patients as referrals wander, so pressure (the list divided by a reference of 40,000 patients) ranges from 0.75 to 1.22 in the rows shown. That variation is what makes the effect learnable: a list that never moved would tell us nothing about how treatments respond to it.

The learning step uses the structure. Because the list changes only through its flows, the treatments in month $t$ are the referrals minus the change in the list, and dividing by capacity gives the effect:

$$\widehat{\text{Treatments}}_t = \text{Referrals}_t - (W_{t+1} - W_t),\qquad \widehat{\text{Effect}}_t = \widehat{\text{Treatments}}_t / \text{Capacity}.$$

We fit a lookup table to these points. A table that draws straight lines between points is linear in its $y$ values, so it can be fitted by ordinary least squares, with a **hat function** basis: each point of the table gets a column that is 1 at its own $x$ and falls to 0 at its neighbours. We place the table's points across the range of pressure seen in the first eight years, and keep the last two for testing.

In [ ]:
train = months[:-1] < 96
pressure = observed[:-1] / 40000
effect_hat = (referrals.to_numpy()[:-1] - np.diff(observed)) / 4000
lo, hi = pressure[train].min(), pressure[train].max()
knots = np.round(np.linspace(lo, hi, 5), 3)


def hats(x, knots):
    x = np.clip(x, knots[0], knots[-1])
    return np.column_stack([np.interp(x, knots, np.eye(len(knots))[j])
                            for j in range(len(knots))])


y_knots = np.linalg.lstsq(hats(pressure[train], knots), effect_hat[train],
                          rcond=None)[0]
learned = "TABLE(Pressure, " + ", ".join(
    f"{x:.3f}, {y:.4f}" for x, y in zip(knots, y_knots)) + ")"
print(f"Pressure seen in training: {lo:.2f} to {hi:.2f}")
print("Learned:", learned)
hybrid = pressure_model(learned).run(inputs={"Referrals": referrals})
err = hybrid.loc[months, "Waiting list"].to_numpy() - observed
print(f"Hybrid model, mean absolute error: training "
      f"{np.abs(err[:96]).mean():.0f}, test {np.abs(err[96:]).mean():.0f}"
      " patients")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
xg = np.linspace(0.5, 2, 200)
true_pts = [0.5, 0.85, 1, 1, 1.5, 1.1, 2, 1.15, 3, 1.17]
ax1.plot(pressure[train], effect_hat[train], "o", ms=3,
         label="inferred from data")
ax1.plot(xg, np.interp(xg, true_pts[0::2], true_pts[1::2]),
         label="true effect")
ax1.plot(xg, np.interp(xg, knots, y_knots), label="learned table")
ax1.axvspan(lo, hi, alpha=0.1, color="grey")
ax1.set(title="The learned effect", xlabel="Pressure (list / 40,000)",
        ylabel="Effect of pressure (dimensionless)")
ax1.legend()
ax2.plot(months, observed / 1000, label="observed")
ax2.plot(months, hybrid.loc[months, "Waiting list"] / 1000,
         label="hybrid model")
ax2.axvline(96, color="grey", lw=1)
ax2.set(title="The hybrid model against the data", xlabel="month",
        ylabel="Waiting list (thousand patients)")
ax2.legend()
plt.tight_layout()
plt.show()

*Figure: left, the effect of pressure on treatments inferred from the data (dots), the true effect and the learned table; the shaded band is the range of pressure seen in training. Right, the hybrid model with the learned table against the data; the vertical line marks the start of the test years.*

The learned table follows the true effect closely over the range of pressure seen in training, 0.75 to 1.39 (at a pressure of 1.0 it gives 0.994, against a true 1.0). Put back into the stock-and-flow structure, it reproduces the list with a mean absolute error of 227 patients in the training years and 711 in the two test years, which it never saw.

Inside the shaded band the table is learned well. Outside it the table is flat, by the convention of `TABLE`, which is a guess, not knowledge: the true effect goes on rising to 1.17 at a pressure of 3. A hybrid model is honest about this because the learned piece is small, has units and a meaning, and can be plotted and argued over. Richer learned pieces exist. **Neural ordinary differential equations** let a neural network stand in for part or all of the rate equations, trained through the simulator [@chen2018]; **physics-informed neural networks** train a network to fit the data while obeying known equations [@raissi2019]. Both go further than a table, and both inherit its weakness outside the data.

**Try it.**

- *Check.* What is the learned effect at a pressure of 1.0? How close is it to the true value?
- *Apply.* Fit the table with only the first four years of data. How wide is the range of pressure seen, and how does the test error change?
- *Extend.* Replace the table by a straight line, `a + b * Pressure`, fitted by least squares to the same points. Which does better on the test years, and why might the line be dangerous at high pressure?

<details><summary>Worked answer</summary>

*Check.* 0.994, within 1% of the true value of 1.0.

*Apply.* The first four years show pressure only from 0.75 to 1.23, so the table has to guess beyond 1.23. The test error rises from 711 to 825 patients.

*Extend.* The line, effect = 0.724 + 0.268 x pressure, does as well on the test years (an error of 710 patients) because the test years stay inside the training range. But at a pressure of 3 it says 1.53, against a true 1.17: it promises that staff go on working ever faster as the list grows. A table flat beyond its ends is a cautious guess; a line is a confident wrong one. Neither replaces asking clinicians what happens at extreme pressure.

</details>

In [ ]:
print("Learned effect at 1.0:", round(float(np.interp(1.0, knots, y_knots)),
                                      3), "(true 1.0)")
early = months[:-1] < 48
lo4, hi4 = pressure[early].min(), pressure[early].max()
k4 = np.round(np.linspace(lo4, hi4, 5), 3)
y4 = np.linalg.lstsq(hats(pressure[early], k4), effect_hat[early],
                     rcond=None)[0]
t4 = "TABLE(Pressure, " + ", ".join(f"{x}, {y:.4f}" for x, y in
                                     zip(k4, y4)) + ")"
h4 = pressure_model(t4).run(inputs={"Referrals": referrals})
e4 = h4.loc[months, "Waiting list"].to_numpy() - observed
print(f"Four years: pressure {lo4:.2f} to {hi4:.2f}, "
      f"test error {np.abs(e4[96:]).mean():.0f}")
a, b = np.linalg.lstsq(np.column_stack([np.ones(train.sum()),
                                        pressure[train]]),
                       effect_hat[train], rcond=None)[0]
hl = pressure_model(f"{a:.4f} + {b:.4f} * Pressure").run(
    inputs={"Referrals": referrals})
el = hl.loc[months, "Waiting list"].to_numpy() - observed
print(f"Line: effect = {a:.3f} + {b:.3f} x pressure; test error "
      f"{np.abs(el[96:]).mean():.0f}; at pressure 3 it says "
      f"{a + 3 * b:.2f} (true 1.17)")

## 4. Testing: a forecasting contest across a policy change

Now the question the data scientist asked. In January 2022 the hospital group launched a waiting-list initiative that bought about 10% more treatment capacity. Train two models on the data up to December 2021, and forecast 2022 and 2023, which we hold back.

- The **pure data model** is an **autoregression**: next month's change in the list is a weighted sum of the last $p$ monthly changes, plus a constant, fitted by least squares. It knows nothing about stocks or policies.
- The **system dynamics model** is the case model: `Waiting list` with `Referrals` and `Treatments = MIN(Capacity, Waiting_list / Minimum_treatment_time)`, referrals and capacity growing at constant rates, and the 2020 fall in activity (capacity down to about a third in March 2020, back over nine months, as hospital activity reports show) as a known input. We calibrate four constants by least squares, with a small Nelder-Mead search written in numpy, as in chapter 9.

The data are synthetic, shaped like published Irish waiting-list counts; see chapter 9.

In [ ]:
y = wl["total"].to_numpy(dtype=float)
t_all = np.arange(len(y), dtype=float)           # months since Jan 2014
TRAIN_END = 96                                   # Jan 2022 is month 96

case = lb.Model.from_spec({
    "name": "Waiting list case", "timeUnit": "month", "T": 119, "dt": 0.25,
    "stocks": [{"id": "wl", "name": "Waiting list", "init": str(y[0]),
                "units": "patients", "nonNegative": True}],
    "flows": [
        {"id": "r", "name": "Referrals", "to": "wl",
         "eq": "Referrals_base * (1 + Referral_growth) ^ (TIME / 12)"
               " * (1 - Demand_cut * STEP(1, Lever_start))",
         "units": "patients/month"},
        {"id": "t", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
         "units": "patients/month"}],
    "vars": [
        {"id": "rb", "name": "Referrals base", "eq": "4000",
         "units": "patients/month"},
        {"id": "rg", "name": "Referral growth", "eq": "0.03",
         "units": "fraction/year"},
        {"id": "cb", "name": "Capacity base", "eq": "3950",
         "units": "patients/month"},
        {"id": "cg", "name": "Capacity growth", "eq": "0.02",
         "units": "fraction/year"},
        {"id": "pan", "name": "Pandemic effect",
         "eq": "TABLE(TIME, 73.9, 1, 74, 0.35, 83, 1)", "units": "1"},
        {"id": "ini", "name": "Initiative size", "eq": "0.10",
         "units": "fraction"},
        {"id": "ls", "name": "Lever start", "eq": "120", "units": "month"},
        {"id": "xc", "name": "Extra capacity", "eq": "0",
         "units": "fraction"},
        {"id": "dc", "name": "Demand cut", "eq": "0", "units": "fraction"},
        {"id": "cap", "name": "Capacity",
         "eq": "Capacity_base * (1 + Capacity_growth) ^ (TIME / 12)"
               " * Pandemic_effect * (1 + Initiative_size * STEP(1, 96))"
               " * (1 + Extra_capacity * STEP(1, Lever_start))",
         "units": "patients/month"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5",
         "units": "month"}]})
KEYS = ["referrals_base", "referral_growth", "capacity_base",
        "capacity_growth"]


def simulate(theta, T=119, **kw):
    d = case.run(T=T, include=["Waiting list"], **dict(zip(KEYS, theta)),
                 **kw)
    return d.loc[np.arange(T + 1.0), "Waiting list"].to_numpy()


def nelder_mead(f, x0, step, iters=500):
    pts = [np.array(x0, float)] + [np.array(x0, float) + np.eye(len(x0))[i]
                                   * step[i] for i in range(len(x0))]
    val = [f(p) for p in pts]
    for _ in range(iters):
        order = np.argsort(val)
        pts, val = [pts[i] for i in order], [val[i] for i in order]
        c = np.mean(pts[:-1], axis=0)
        xr = 2 * c - pts[-1]
        fr = f(xr)
        if fr < val[0]:
            xe = 3 * c - 2 * pts[-1]
            fe = f(xe)
            pts[-1], val[-1] = (xe, fe) if fe < fr else (xr, fr)
        elif fr < val[-2]:
            pts[-1], val[-1] = xr, fr
        else:
            xc = (c + pts[-1]) / 2
            fc = f(xc)
            if fc < val[-1]:
                pts[-1], val[-1] = xc, fc
            else:
                pts = [pts[0]] + [(p + pts[0]) / 2 for p in pts[1:]]
                val = [val[0]] + [f(p) for p in pts[1:]]
    return pts[int(np.argmin(val))]


def calibrate(end):
    sse = lambda th: ((simulate(th, T=end - 1) - y[:end]) ** 2).sum()
    return nelder_mead(sse, [3800, 0.02, 3800, 0.01],
                       [200, 0.01, 200, 0.01])


theta = calibrate(TRAIN_END)
print({k: round(float(v), 4) for k, v in zip(KEYS, theta)})

The fit gives referrals of 3,553 patients/month growing 6.4% a year and capacity of 3,508 patients/month growing 5.4% a year. These need not be the true values: the stock alone pins down mostly the difference between referrals and capacity, and the 2020 fall in activity is what pins down the level of capacity (chapter 9 discusses this kind of identification). For forecasting the list, the difference is what counts.

The autoregression is fitted to monthly changes, and forecasts are built by feeding each predicted change back in. We fit orders $p$ of 1, 2, 3, 6 and 12 months.

In [ ]:
def ar_fit(x, p):
    rows = np.column_stack([np.ones(len(x) - p)]
                           + [x[p - k - 1:len(x) - k - 1] for k in range(p)])
    return np.linalg.lstsq(rows, x[p:], rcond=None)[0]


def ar_forecast(x, coef, h):
    hist = list(x)
    for _ in range(h):
        hist.append(coef[0] + sum(c * hist[-k - 1]
                                  for k, c in enumerate(coef[1:])))
    return np.array(hist[len(x):])


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


test = slice(TRAIN_END, len(y))
h = len(y) - TRAIN_END
dy = np.diff(y[:TRAIN_END])
forecasts = {}
for p in (1, 2, 3, 6, 12):
    forecasts[f"autoregression, p = {p}"] = \
        y[TRAIN_END - 1] + np.cumsum(ar_forecast(dy, ar_fit(dy, p), h))
forecasts["no change (last value)"] = np.full(h, y[TRAIN_END - 1])
forecasts["system dynamics, policy ignored"] = \
    simulate(theta, initiative_size=0)[test]
forecasts["system dynamics, with the initiative"] = simulate(theta)[test]
score = pd.Series({k: rmse(v, y[test]) for k, v in forecasts.items()})
print("Error over 2022 and 2023 (RMSE, patients):")
print(score.round(0).sort_values().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
dates = wl["month"]
ax.plot(dates, y / 1000, label="data", lw=2.5)
for k in ("autoregression, p = 2", "system dynamics, policy ignored",
          "system dynamics, with the initiative"):
    ax.plot(dates[test], forecasts[k] / 1000, label=k)
ax.axvline(dates[TRAIN_END], color="grey", lw=1)
ax.set(title="Forecasting 2022 and 2023 from data up to 2021",
       xlabel="month", ylabel="Waiting list (thousand patients)")
ax.legend(loc="upper left")
plt.show()

*Figure: the waiting list (data) and three forecasts made at the end of 2021 (vertical line). The autoregression extends the climb it learned; the system dynamics model with the initiative bends with the data.*

Over 2022 and 2023 the system dynamics model told about the initiative has the smallest error, 872 patients. Every autoregression extends the post-pandemic climb and misses by about 4,000 to 4,900 patients; the best, with $p$ = 2, ends 2023 at 76,990 against an actual 69,927. The system dynamics model without the initiative is worst of all, 6,686: structure alone does not help if the policy is left out. What wins is structure plus knowledge of the policy, and only a structural model has a place to put that knowledge.

**Where the data model wins.** Repeat the contest in quiet times: train on 2014 to 2017 and forecast 2018 and 2019, when no policy changed. Over a two-year horizon the two approaches come close. One month ahead, where the autoregression can use last month's actual change, it does a little better: it picks up short-term patterns the system dynamics model left out on purpose.

In [ ]:
QUIET = 48                                         # Jan 2018
theta_q = calibrate(QUIET)
sd_q = simulate(theta_q)
dq = np.diff(y[:QUIET])
win = slice(QUIET, QUIET + 24)
ar_q = y[QUIET - 1] + np.cumsum(ar_forecast(dq, ar_fit(dq, 1), 24))
print(f"Two years ahead: system dynamics {rmse(sd_q[win], y[win]):.0f}, "
      f"autoregression p = 1 {rmse(ar_q, y[win]):.0f}")
one_ar, one_sd = [], []
coef = ar_fit(dq, 1)
for t in range(QUIET, QUIET + 24):
    one_ar.append(y[t - 1] + ar_forecast(np.diff(y[:t]), coef, 1)[0])
    one_sd.append(y[t - 1] + sd_q[t] - sd_q[t - 1])
print(f"One month ahead: system dynamics {rmse(one_sd, y[win]):.0f}, "
      f"autoregression p = 1 {rmse(one_ar, y[win]):.0f}, "
      f"no change {rmse(y[QUIET - 1:QUIET + 23], y[win]):.0f}")
print("Quiet-time calibration:",
      {k: round(float(v), 3) for k, v in zip(KEYS, theta_q)})

In quiet times the two are close: over two years the system dynamics model misses by 1,167 patients and the autoregression by 1,331; one month ahead the autoregression is slightly better (177 against 190), and both beat "no change" (280). Look also at the quiet-time calibration: both growth rates come out at 44.5% a year, which is absurd. Without the 2020 shock the data pin down only their difference, so a model can forecast well with parameters that mean nothing. A good fit is not a correct model.

The lesson is not that one family of models is better. A data model learns the patterns in its training data and is good at continuing them; it has no way to represent a change it has never seen. A structural model represents the mechanism through which a policy acts, so it can be told about the policy; that is why it is the right tool for "what if we act?". When nothing changes and the horizon is short, the data model is a strong, cheap benchmark, and every system dynamics forecast should be compared with one.

**Try it.**

- *Check.* Why does the "no change" forecast do so well over 2022 and 2023?
- *Apply.* Suppose the initiative had been 20%, not 10%. Re-run the system dynamics forecast with `initiative_size=0.2` and compare its end-2023 list with the 10% forecast. What would the autoregression say?
- *Extend.* Give the autoregression a fair chance: add an input column that is 1 from January 2022 (a policy dummy) to its regression. Can it learn the effect of the initiative from the training data? Why not?

<details><summary>Worked answer</summary>

*Check.* Because the list barely moved: it rose by only 1,375 patients over the two years, because the initiative happened to bring treatments close to referrals. "No change" was right by coincidence of policy, and it would have said the same for any initiative, or none.

*Apply.* With a 20% initiative the system dynamics model ends 2023 at 55,835 patients instead of 68,791. The autoregression says 76,990 in both cases: it has no input through which the size of the initiative could act.

*Extend.* The dummy is zero in every training month (there were 0 months with the initiative before 2022), so its column carries no information and its coefficient cannot be estimated. A data model can learn the effect of a policy only from past episodes of that policy. A structural model can reason about a new policy from how the system works.

</details>

In [ ]:
print("Change in the list during 2022-2023 (data):", y[-1] - y[TRAIN_END - 1])
s10, s20 = simulate(theta), simulate(theta, initiative_size=0.2)
print(f"End of 2023: 10% initiative {s10[-1]:.0f}, 20% initiative "
      f"{s20[-1]:.0f}, autoregression p = 2 "
      f"{forecasts['autoregression, p = 2'][-1]:.0f} in both cases")
dummy = (t_all[1:TRAIN_END] >= 96).astype(float)
print("Months with the dummy switched on in the training data:",
      int(dummy.sum()))

## 5. Policy and use: surrogate models

Policy search, calibration and uncertainty analysis need thousands of runs. When one run takes minutes (a large model, an agent-based model), a **surrogate**, also called a **metamodel** or **emulator**, helps: a cheap statistical model fitted to a few hundred runs of the real one, used to search, and then checked against the real model at the answer.

We use the calibrated case model from section 4, run on to the end of 2028, with two levers from January 2024: `Extra capacity` (a fraction, 0 to 0.15) and `Demand cut` (a fraction, 0 to 0.10). The outcome is the list at the end of 2028. We fit a quadratic surrogate to 40 runs and test it on 400 others, all done with `run_batch`.

In [ ]:
calibrated = dict(zip(KEYS, theta))
T_END = 179                                       # December 2028


def outcome(levers):
    settings = pd.DataFrame(levers).assign(**calibrated)
    out = case.run_batch(settings, T=T_END, include=["Waiting list"])
    return out[out.time == T_END].sort_values("run")["value"].to_numpy()


def design(n, seed):
    g = np.random.default_rng(seed)
    return {"extra_capacity": g.uniform(0, 0.15, n),
            "demand_cut": g.uniform(0, 0.10, n)}


def features(lev):
    a, b = np.asarray(lev["extra_capacity"]), np.asarray(lev["demand_cut"])
    return np.column_stack([np.ones_like(a), a, b, a * a, a * b, b * b])


fit_runs, test_runs = design(40, SEED), design(400, SEED + 1)
y_fit, y_test = outcome(fit_runs), outcome(test_runs)
beta = np.linalg.lstsq(features(fit_runs), y_fit, rcond=None)[0]
pred = features(test_runs) @ beta
r2 = 1 - ((pred - y_test) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()
print(f"Surrogate on 400 unseen runs: R^2 = {r2:.4f}, largest error "
      f"{np.abs(pred - y_test).max():.0f} patients")

The quadratic surrogate explains 98.5% of the variation in the 400 unseen runs ($R^2$ = 0.985), but its largest error is 9,153 patients: good on average, poor in places.

Now search. Suppose each percentage point of extra capacity costs EUR 3.5 million over the five years and each point of demand cut EUR 2 million (illustrative prices). Which is the cheapest combination that brings the list below 50,000 by the end of 2028? The surrogate scores a fine grid of 10,201 combinations at once; the real model then checks the winner.

In [ ]:
xa, xb = np.meshgrid(np.linspace(0, 0.15, 101), np.linspace(0, 0.10, 101))
cand = {"extra_capacity": xa.ravel(), "demand_cut": xb.ravel()}
guess = features(cand) @ beta
cost = 350 * cand["extra_capacity"] + 200 * cand["demand_cut"]  # EUR m
ok = guess < 50000
i = int(np.argmin(np.where(ok, cost, np.inf)))
pick = {k: v[i:i + 1] for k, v in cand.items()}
real = outcome(pick)[0]
print(f"Surrogate pick: extra capacity {pick['extra_capacity'][0]:.1%}, "
      f"demand cut {pick['demand_cut'][0]:.1%}, cost EUR {cost[i]:.1f} m")
print(f"Surrogate says {guess[i]:.0f}; the real model says {real:.0f}")

The surrogate's cheapest pick is no extra capacity and a 6.8% cut in demand, for EUR 13.6 million. It predicts 49,724 patients; the real model says 51,345, which misses the target. An optimiser drives towards the boundary of what the surrogate thinks is allowed, so a small surrogate error is enough to fail. The fix is cheap: shortlist the candidates the surrogate likes, with a safety margin, and let the real model decide among them.

In [ ]:
near = np.flatnonzero(guess < 50000 + 3000)        # a safety margin
short = near[np.argsort(cost[near])][:300]         # the 300 cheapest
checked = outcome({k: v[short] for k, v in cand.items()})
j = short[np.flatnonzero(checked < 50000)[0]]
print(f"Checked pick: extra capacity {cand['extra_capacity'][j]:.1%}, "
      f"demand cut {cand['demand_cut'][j]:.1%}, cost EUR {cost[j]:.1f} m, "
      f"real model {checked[short == j][0]:.0f}")

The checked pick is a 7.1% cut in demand and no extra capacity, for EUR 14.2 million, and the real model confirms 49,982 patients. Under these illustrative prices demand management is the cheaper lever; with different prices the answer would change, which is exactly the kind of question the surrogate makes quick to explore.

With a model this small the surrogate is not needed: `run_batch` would score all 10,201 combinations directly in a second or two. The method earns its place when each run is slow. Two rules keep it safe: test the surrogate on runs it was not fitted to, and always confirm the chosen policy with the real model, because a surrogate is least accurate exactly where an optimiser likes to go, at the edges of the region.

**Try it.**

- *Check.* Is the surrogate's pick at the edge of the 50,000 constraint? Why would that be expected?
- *Apply.* Fit a linear surrogate (drop the three squared and product terms). How do its $R^2$ and largest error on the 400 test runs compare?
- *Extend.* Check the surrogate's pick with the real model at the edges of its range: what does the surrogate predict, and the model give, for 15% extra capacity and a 10% demand cut together?

<details><summary>Worked answer</summary>

*Check.* Yes: 49,982 patients against a limit of 50,000. Cost rises with both levers and the list falls with both, so the cheapest policy that meets the target sits on its boundary.

*Apply.* The linear surrogate has $R^2$ = 0.951 and a largest error of 21,270 patients, against 0.985 and 9,153 for the quadratic one. The curvature matters.

*Extend.* At 15% extra capacity and a 10% demand cut the surrogate predicts -7,304 patients, an impossible negative list; the real model gives 4,011, held up by the limit `Waiting_list / Minimum_treatment_time`. The polynomial knows nothing about that floor.

</details>

In [ ]:
lin = np.linalg.lstsq(features(fit_runs)[:, :3], y_fit, rcond=None)[0]
pl = features(test_runs)[:, :3] @ lin
r2l = 1 - ((pl - y_test) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()
print(f"Linear surrogate: R^2 = {r2l:.4f}, largest error "
      f"{np.abs(pl - y_test).max():.0f}")
corner = {"extra_capacity": np.array([0.15]), "demand_cut": np.array([0.10])}
print(f"Corner: surrogate {(features(corner) @ beta)[0]:.0f}, "
      f"model {outcome(corner)[0]:.0f}")

## 6. A caution: overfitting, leakage and explainability

Three failures account for most disappointing results when machine learning meets system dynamics.

**Overfitting.** A flexible model can match the training data ever more closely while forecasting worse. The autoregression shows it: as the order rises, the in-sample error falls, but the forecast error over 2022 and 2023 does not follow.

In [ ]:
P0 = 24                     # every order is fitted to the same months
rows = []
for p in (1, 2, 3, 6, 12, 18, 24):
    X = np.column_stack([np.ones(len(dy) - P0)]
                        + [dy[P0 - k - 1:len(dy) - k - 1] for k in range(p)])
    coef = np.linalg.lstsq(X, dy[P0:], rcond=None)[0]
    in_sample = rmse(X @ coef, dy[P0:])
    fc = y[TRAIN_END - 1] + np.cumsum(ar_forecast(dy, coef, h))
    rows.append({"order p": p, "coefficients": p + 1,
                 "training error (change/month)": in_sample,
                 "forecast error 2022-23 (patients)": rmse(fc, y[test])})
print(pd.DataFrame(rows).round(0).to_string(index=False))

Every order is fitted to the same months (from 2016 on, after the first 24 changes), so the errors differ from those of section 4. The training error falls steadily, from 414 to 369 patients/month, as the order rises from 1 to 24. The forecast error does not follow: it stays between about 5,500 and 5,900 patients up to order 18, then jumps to 20,022 at order 24, the model with the best fit.

**Leakage** is information from the test period that slips into training. It makes a method look better than it is, and it is easy to do by accident: calibrate on the whole series and then "test" on its last two years; or choose among models by their test score, which turns the test set into training data. Here is the first kind.

In [ ]:
theta_all = calibrate(len(y))
print(f"Calibrated on data up to 2021: test error "
      f"{rmse(simulate(theta)[test], y[test]):.0f}")
print(f"Calibrated on all data (leak): test error "
      f"{rmse(simulate(theta_all)[test], y[test]):.0f}")

Calibrated on all the data, the "test" error falls from 872 to 373 patients. Nothing has improved: the model has simply seen the answers. In a report this number would overstate accuracy by more than half.

**Explainability.** A decision maker who asks "why does the list keep growing?" can be answered from the system dynamics model in its own terms: referrals grow faster than capacity, and the list is the accumulated difference. An autoregression's coefficients have no such reading, and a neural network's even less. Hybrid models sit in between: keep the learned parts small, give them units and a meaning, plot them, and test them under extreme conditions as in chapter 8 (Testing your model). A checklist before trusting any learned component:

1. Was it tested on data it never saw, split by time, not at random?
2. Was anything about the test period used in training, scaling or model choice?
3. Does it behave sensibly outside the range of its training data?
4. Can you say in one sentence what it does, in the units of the problem?

**Try it.**

- *Check.* In the overfitting table, which order has the lowest training error, and what is its forecast error?
- *Apply.* Make the second kind of leakage: choose the autoregression order by the 2022 to 2023 error, then report that error as "the" forecast accuracy. Why is the number now biased?
- *Extend.* Split the training data into 2014 to 2019 for fitting and 2020 to 2021 for choosing the order (a validation set). Which order is chosen, and what is its honest error on 2022 to 2023?

<details><summary>Worked answer</summary>

*Check.* Order 24, with a training error of 369 patients/month and a forecast error of 20,022 patients, the worst in the table.

*Apply.* Choosing on the test set picks order 18, with an error of 5,465. That number is biased downwards because the order was chosen to make it small: it is no longer a test, and a new two years of data would almost certainly show a larger error.

*Extend.* With 2020 and 2021 as a validation period, order 24 is chosen and its honest error on 2022 and 2023 is 20,022 patients. The validation years were the pandemic, unlike anything before or after, and they rewarded a model with a long memory of the shock. A validation set must resemble the future you want to forecast; and an honest number is sometimes an ugly one.

</details>

In [ ]:
ov = pd.DataFrame(rows).set_index("order p")
print("Lowest training error at p =",
      ov["training error (change/month)"].idxmin())
print("Best order chosen on the test set:",
      ov["forecast error 2022-23 (patients)"].idxmin())
VAL = 72                                           # Jan 2020
dv = np.diff(y[:VAL])
val_err = {}
for p in (1, 2, 3, 6, 12, 18, 24):
    fc = y[VAL - 1] + np.cumsum(ar_forecast(dv, ar_fit(dv, p),
                                            TRAIN_END - VAL))
    val_err[p] = rmse(fc, y[VAL:TRAIN_END])
p_val = min(val_err, key=val_err.get)
honest = y[TRAIN_END - 1] + np.cumsum(ar_forecast(dy, ar_fit(dy, p_val), h))
print("Order chosen on 2020-21:", p_val, "| honest error on 2022-23:",
      round(rmse(honest, y[test])))

## Check it

Habits for any project that mixes learning and structure:

1. **Split by time.** Train on the past, test on the future, never at random: a random split lets the model see the months around each test month.
2. **Benchmark.** Compare every forecast with "no change" and with a simple autoregression. If the structural model cannot beat them in quiet times, find out why before using it for policy.
3. **Units and extremes for learned parts.** A learned effect must be dimensionless if it multiplies a rate, and must stay sensible at zero and at very high inputs.

In [ ]:
print("Training months end before the first test month:",
      TRAIN_END - 1 < test.start)
print("Benchmarks over 2022-23:",
      {k: round(score[k]) for k in ("no change (last value)",
                                    "system dynamics, with the initiative")})
for w in (0, 200000):
    e = pressure_model(learned).with_init("Waiting list", w).run(
        T=0, inputs={"Referrals": referrals})["Effect of pressure"].iloc[0]
    print(f"Learned effect with a list of {w}: {e:.3f}")

## Running case

This chapter adds no new structure to the waiting-list model. It puts the model next to machine learning at each stage: an LLM-style draft of the causal structure checked against interview notes; a hybrid version whose effect of pressure on treatments is learned from data; a forecasting contest in which the case model, calibrated to data up to 2021 and told about the 2022 initiative, beats an autoregression by a wide margin; and a surrogate of the calibrated model for a fast policy search to 2028.

## Tell the decision maker

"A forecast built only from past data would have told you the list would keep climbing through 2022 and 2023; it cannot know about the initiative you funded. Our model can, and it correctly showed the climb levelling off. For questions about what to do, use a model of how the system works; for next month's figure in quiet times, a simple trend is about as good."

## Going further

- **Importing existing models.** Many published and in-house models are in Vensim or XMILE (Stella) format. PySD [@pysd2022] translates them to Python, so they can be run, calibrated and combined with the methods of this chapter.
- **Structure discovery.** The `pysindy` package implements SINDy [@brunton2016] with many libraries, derivative estimators and sparse solvers; try it on two-stock molecules such as the Diffusion molecule.
- **Neural differential equations and physics-informed learning.** See @chen2018 and @raissi2019, and look for "universal differential equations", which embed small networks inside known equations, the same idea as the hybrid model of section 3.
- **LLMs in modelling.** Use them for drafting, translation between diagram and equations, and documentation, always with evidence checks like those of section 1, and with the prompt, model version and date recorded.

## Mini-project

The waiting-list team wants to know whether the effect of pressure on treatments is real. Build a hybrid version of the case model in which `Treatments = MIN(Capacity * Effect_of_pressure, Waiting_list / Minimum_treatment_time)`, with a learned table of three points, and calibrate it, with the four constants of section 4, to the data up to 2021 using `nelder_mead`. Compare its forecast of 2022 and 2023 with the plain case model and with the best autoregression, and write three sentences for the team: is the effect supported by the data, what does it change in the forecast, and how confident are you?

## Further reading

On calibration as a test of structure see @oliva2003, and on validity @barlas1996. The comparison of differential-equation and individual-level models of diffusion in @rahmandad2008 is a model for careful comparison of modelling approaches. For the methods introduced here, start with @brunton2016 for sparse identification, @chen2018 for neural ordinary differential equations and @raissi2019 for physics-informed neural networks; @pysd2022 describes PySD.